In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# print(os.listdir(root_dir))

In [ ]:
# libraries
!pip install mne --quiet
import mne, os, numpy as np, matplotlib.pyplot as plt
import glob

In [ ]:
# path
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/'


In [ ]:
sessions = ['3_001.set', '3_002.set']
event_tags = ['Wd2E', 'Wd2N'] # trial markers of interest
tmin, tmax = -0.10, 0.40      # epochs of 500ms

In [ ]:
epochs_all = []
for ses in sessions:

    # Read & preprocess
    raw = mne.io.read_raw_eeglab(os.path.join(root_dir, ses), preload=True)
    raw.filter(l_freq=0.1, h_freq=None) # 0.1 Hz
    raw.notch_filter(freqs=60.) # notch 60 Hz

    # Extract events of interest
    events, annot_id = mne.events_from_annotations(raw) # dict of all markers
    print(f"Events for session {ses}:")
    print(events)

    event_id = {k:v for k,v in annot_id.items() if k in event_tags}

    epochs = mne.Epochs(raw, events, event_id,
                        tmin=tmin, tmax=tmax, baseline=(None, 0),
                        preload=True, picks='eeg', reject_by_annotation=True)
    epochs_all.append(epochs)

In [ ]:
# combine epochs
epochs_combined = mne.concatenate_epochs(epochs_all)

In [ ]:
regions = {
    'Ant-Left'  : 'F7',   'Ant-Med'  : 'Fz',  'Ant-Right' : 'F8',
    'Cen-Left'  : 'C3',   'Cen-Med'  : 'Cz',  'Cen-Right' : 'C4',
    'Post-Left' : 'T5',   'Post-Med' : 'Pz',  'Post-Right': 'T6'
}

In [ ]:
# helper: prettify legend handles once
color_map = {'Wd2E':'tab:blue', 'Wd2N':'tab:orange'}
handles = [plt.Line2D([0],[0], color=c, lw=2) for c in color_map.values()]

# loop over every subject folder
for folder in sorted([f for f in os.listdir(root_dir) if f.isdigit()]):
    part_path = os.path.join(root_dir, folder)
    set_files = sorted(glob.glob(os.path.join(part_path, f'{folder}_*.set')))
    if not set_files:
        print(f'⚠️ No .set files in {part_path} – skipped.')
        continue

    epochs_list = []
    for fset in set_files:
        raw = mne.io.read_raw_eeglab(fset, preload=True, verbose=False)

        # preprocessing
        raw.filter(l_freq=0.1, h_freq=None, verbose=False)
        raw.notch_filter(freqs=60., verbose=False)

        # events → epochs
        events, annot_id = mne.events_from_annotations(raw, verbose=False)
        event_id = {k:v for k,v in annot_id.items() if k in event_tags}
        if not event_id:
            print(f'⚠️{os.path.basename(fset)} lacks Wd2 markers – skipped.')
            continue

        epochs = mne.Epochs(raw, events, event_id,
                            tmin=tmin, tmax=tmax,
                            baseline=(None, 0),
                            picks='eeg', preload=True,
                            reject_by_annotation=True, verbose=False)
        epochs_list.append(epochs)

    if not epochs_list:
        print(f'⚠️No usable epochs in {folder}.')
        continue

    # concatenate all sessions for this participant
    epo_all = mne.concatenate_epochs(epochs_list)

    # average per condition
    evokeds = {tag: epo_all[tag].average() for tag in event_tags}

    fig, axes = plt.subplots(3, 3, figsize=(14,10), sharex=True, sharey=True)

    # Get the list of available channel names from the evoked data
    available_channels = evokeds[event_tags[0]].ch_names

    for i, (region, ch) in enumerate(regions.items()):
        r, c = divmod(i, 3)
        ax   = axes[r][c]

        # Check if the channel from the regions dictionary exists in the current data
        if ch in available_channels:
            for tag in event_tags:
                # Pick the channel by its name if it exists
                y = evokeds[tag].copy().pick(ch).data.squeeze() * 1e6     # µV
                ax.plot(evokeds[tag].times*1e3, y, color=color_map[tag], label=tag)
            ax.axvline(0, color='k', lw=.8)
            ax.set_title(f'{region} ({ch})', fontsize=10)
        else:
            # If the channel does not exist, print a warning and set the title to indicate it's missing
            print(f"⚠️ Channel '{ch}' for region '{region}' not found in data for participant {folder}.")
            ax.set_title(f'{region} ({ch}) (Missing)', fontsize=10, color='red')
            ax.text(0.5, 0.5, 'Channel Not Found', horizontalalignment='center', verticalalignment='center', transform=ax.transAxes, color='gray')


        if r == 2: ax.set_xlabel('Time (ms)')
        if c == 0: ax.set_ylabel('Amplitude (µV)')

    fig.suptitle(f'Participant {folder}:  ERP overview (Wd2E vs Wd2N)', fontsize=14)
    fig.legend(handles, color_map.keys(), loc='lower center', ncol=2)
    plt.tight_layout(rect=[0,0.05,1,0.96])